# Skyborne - train the white-tent YOLO11n model (Google Colab)
Runtime -> Change runtime type -> **T4 GPU**. Then run the cells in order.
Upload `tents_dataset.zip` (made by `tools/make_yolo_dataset.py`, add your real labelled photos into the same folders first).

In [ ]:
!pip -q install ultralytics onnx onnxslim
from google.colab import files
up = files.upload()   # choose tents_dataset.zip
!unzip -q -o tents_dataset.zip -d /content/tents
!ls /content/tents

In [ ]:
open('/content/tents/data.yaml','w').write('path: /content/tents\ntrain: images/train\nval: images/val\nnames:\n  0: white_tent\n')
from ultralytics import YOLO
model = YOLO('yolo11n.pt')   # COCO-pretrained, 2.6 M parameters
model.train(data='/content/tents/data.yaml', epochs=60, imgsz=640, batch=32, patience=20,
            degrees=180, flipud=0.5, fliplr=0.5, mosaic=1.0, scale=0.4, hsv_v=0.3, close_mosaic=10)

In [ ]:
best = YOLO('runs/detect/train/weights/best.pt')
m = best.val(data='/content/tents/data.yaml', imgsz=640)
print('mAP50', m.box.map50, 'mAP50-95', m.box.map)
path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
import shutil; shutil.copy(path, 'tent_yolo.onnx')
files.download('tent_yolo.onnx')   # put it in Skyborne/models/tent_yolo.onnx